In [13]:
import pandas as pd
import os

folder = r"C:\Users\Prabha.Sivasankar-C\OneDrive - Seatrium Ltd\Documents\CostAnlaysis\Material Cost-Alan"
source_file = "ZMM_POREPORT_All harmonized projects_R1_20260513 (1).xlsx"
size_sch_file="Material Naming_R7_20260702 - Copy.xlsx"
pipe_chart_file="APP-PipeChart-AverageWall.xlsx"
OUT_XLSX  = r'C:\Users\Prabha.Sivasankar-C\OneDrive - Seatrium Ltd\Documents\CostAnlaysis\Material Cost-Alan\extracted_PO_ZMM_POREPORT_All harmonized projects.xlsx'


In [23]:
def load_df(folder,file,sheetName,skipRows):

    df = pd.read_excel(os.path.join(folder,file),sheet_name=sheetName if sheetName else 0,skiprows=skipRows if skipRows else 0)

    return df

In [3]:
df_cost = load_df(folder,source_file,'Sheet2')
df_size=load_df(folder,size_sch_file,'PSCM_Size Ref_Code ')
df_sch=load_df(folder,size_sch_file,'PSCM_Schedule Ref_Code ')
df_pipe_chart=load_df(folder,pipe_chart_file)

ValueError: Worksheet named 'PSCM_Schedule Ref_Code' not found

In [24]:
df_pipe_chart=load_df(folder,pipe_chart_file,'PIPE CHART',3)

In [14]:
df_size=load_df(folder,size_sch_file,'PSCM_Size Ref_Code ')
df_sch=load_df(folder,size_sch_file,'PSCM_Schedule Ref_Code ')
df_pipe_chart=load_df(folder,pipe_chart_file,'PIPE CHART',3)


In [17]:
df_cost_pipes = df_cost[df_cost['Material Group'] == '910-2'].copy()

In [29]:
df_pipe_chart.columns

Index([                                'NOM',
                                      'O.D.',
                                           5,
                                          10,
                                          20,
                                          30,
                                          40,
                                       'STD',
                                          60,
                                          80,
                                        'XH',
                                         100,
                                         120,
                                         140,
                                         160,
                                       'XXH',
       '<<<<<  HEAVIER THAN SCHEDULED >>>>>',
                               'Unnamed: 17',
                               'Unnamed: 18',
                               'Unnamed: 19',
                               'Unnamed: 20',
                               'Un

Extracting size, size description, scheduel and schedule description

In [20]:

df_cost_pipes[['extracted_Size', 'extracted_Schedule']] = df_cost_pipes['Material'].str.extract(
    r'-(\d{3})-(\d{3})$'
)

df_cost_pipes['Size_Description'] = df_cost_pipes['extracted_Size'].map(
    df_size.set_index('Size Code')['Size Description']
)
df_cost_pipes['Schedule Description'] = df_cost_pipes['extracted_Schedule'].map(
    df_sch.set_index('Schedule Code')['Schedule Description']
)


Extracting thickness based od and sc

In [57]:
df_pipe.columns = [str(c).strip().upper() for c in df_pipe.columns]
for c in df_pipe.columns:
    print(repr(c))




'NOM'
'5'
'10'
'20'
'30'
'40'
'STD'
'60'
'80'
'XH'
'100'
'120'
'140'
'160'
'XXH'
'<<<<<  HEAVIER THAN SCHEDULED >>>>>'
'UNNAMED: 17'
'UNNAMED: 18'
'UNNAMED: 19'
'UNNAMED: 20'
'UNNAMED: 21'
'UNNAMED: 22'
'UNNAMED: 23'
'UNNAMED: 24'


In [81]:
import re
from fractions import Fraction
# df_pipe = df_pipe_chart.set_index("O.D.")
df_pipe_chart["NOM"]=df_pipe_chart["NOM"].astype(float)
df_pipe_chart["NOM"]=df_pipe_chart["NOM"].map(lambda x:f"{x:.3f}")
df_pipe=df_pipe_chart.set_index("NOM")

df_pipe.columns = [str(c).strip().upper() for c in df_pipe.columns]
# df_pipe.index = [str(x).strip() for x in df_pipe.index]

# df_pipe.index = [
#     f"{float(x):.3f}" if str(x).replace('.', '', 1).isdigit() else str(x).strip()
#     for x in df_pipe.index
# ]

def normalize_sch(sch):
    sch=str(sch).strip().upper()
    m=re.search(r'(\d+)',sch)
    if m:
        return str((m.group(1)))
    return sch
def size_to_nom(size):
    size=str(size).replace('"','').strip()
    try:
        parts=size.split()
        if(len(parts)==2):
            value=float(parts[0])+float(Fraction(parts[1]))
        else:
            value=float(Fraction(parts[0]))
        return round(value,3)
    except:
        return None

def get_thickness_od(row):
    nom_value=size_to_nom(row["Size_Description"])
    
    try:
        nom = str(f"{float(nom_value):.3f}")
    except (ValueError, TypeError):
        nom = str(nom_value)

    # nom=f"{size_to_nom(row['Size_Description']):.3f}"
    sch=str(normalize_sch(row["Schedule Description"]))
    # print(nom,sch)
    
    try:
        thk = df_pipe.loc[nom, sch]
        od=df_pipe.loc[nom, 'O.D.']
        print("Found:", od)
        return pd.Series({'Thickness': thk,'OD': od})
    except Exception as e:
        print(
            f"nom={nom!r}, sch={sch!r}, "
            f"error={type(e).__name__}: {e}"
        )
        return None
df_cost_pipes[['Thickness', 'OD']] = df_cost_pipes.apply(
    get_thickness_od,
    axis=1
)

df_cost_pipes['Weight'] = (
    0.02466
    * (
        pd.to_numeric(df_cost_pipes['OD'], errors='coerce')
        - pd.to_numeric(df_cost_pipes['Thickness'], errors='coerce')
    )
    * pd.to_numeric(df_cost_pipes['Thickness'], errors='coerce')
)


Found: 6.625
Found: 2.375
Found: 4.5
Found: 1.315
Found: 10.75
Found: 8.625
Found: 12.75
Found: 16.0
Found: 2.375
Found: 3.5
Found: 4.5
Found: 6.625
Found: 8.625
Found: 1.315
Found: 1.9
Found: 10.75
nom='None', sch='NAN', error=KeyError: 'NAN'
nom='None', sch='NAN', error=KeyError: 'NAN'
nom='None', sch='NAN', error=KeyError: 'NAN'
nom='None', sch='NAN', error=KeyError: 'NAN'
nom='None', sch='NAN', error=KeyError: 'NAN'
nom='None', sch='NAN', error=KeyError: 'NAN'
Found: 6.625
Found: 2.375
Found: 1.315
Found: 12.75
Found: 8.625
Found: 6.625
Found: 2.375
Found: 1.315
Found: 10.75
Found: 8.625
Found: 12.75
Found: 16.0
Found: 1.9
Found: 2.375
Found: 3.5
Found: 4.5
Found: 6.625
Found: 8.625
Found: 1.315
Found: 1.05
Found: 10.75
Found: 1.9
Found: 14.0
Found: 2.375
Found: 3.5
Found: 4.5
Found: 6.625
Found: 1.315
Found: 1.9
Found: 10.75
Found: 10.75
Found: 14.0
Found: 8.625
Found: 6.625
Found: 10.75
Found: 2.375
Found: 1.315
Found: 1.05
Found: 10.75
Found: 1.315
Found: 1.9
Found: 2.875
Found:

In [76]:
df_cost_pipes.columns

Index(['Project Description', 'Project Code', 'Plant', 'Purch. organization',
       'Description', 'Order Type', 'PurchasingDocument', 'Item',
       'Deletion Indicator', 'Our Reference', 'Purchase Requisition',
       'Created On', 'Supplier/Supplying Plant', 'Material Group',
       'Material Group Desc.', 'Material', 'Material Description 1',
       'Material Description 2', 'Material Description 3',
       'Material Description 4', 'Material Description 5',
       'Material Description 6', 'Material Description 7', 'Order Quantity',
       'Order Unit', 'Currency', 'Net Price', 'Net Price Unit',
       'Net Price Order unit', 'Net Price Value', 'Currency.1',
       'Exchange Rate', 'Net Order Value - Spot Rate', 'Project Rate',
       'Net Order Value - Project Rate', 'WBS element',
       'WBS Element Description', 'GL account', 'Purchase Order Header Text',
       'Purchase Requisition Header Text', 'Still to be delivered (Qty)',
       'Still to be delivered (UOM)', 'Still to 

In [83]:
selected_cols = ['WBS element','WBS Element Description','Item','Material','Material Description 1','extracted_Size','Size_Description', 'extracted_Schedule','Schedule Description','Thickness', 'OD','Weight']

df_cost_pipes[selected_cols].to_excel(
    OUT_XLSX,
    index=False
)
